# plot_01d. ERA5: observed change (figures)

Drawn from what calc_03c saved, so nothing slow runs: whether ERA5's 1979–2013 trend could have come from each experiment (section 1), the rates of record highs and lows (2), the detection maps (3) and how much of each forcing's response is observed (4). The model evaluation is plot_01a to plot_01c.

## Setup

In [1]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

from plotting_modules import formatting
from plotting_modules.constants import SEASONS
from extant import era5_evaluation as ev, observed_change as oc
from extant.config import VARIABLES
from extant.datatree import tree_to_dataset
from extant.logger import setup_logging
from extant.plots import antarctica, charts, era5_evaluation as evp, forced_response as fr, observed_change as ocp
from extant.plots import response_change as rcp

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [2]:
%matplotlib inline

## Settings

In [3]:
#(t): As in calc_03c: the variable (its name and units), and the grid point and its label
VARIABLE = VARIABLES['tas']
SAMPLE_LAT_LON = dict(lat=-82.5, lon=120.0)
POINT_LABEL = formatting.format_latlon(SAMPLE_LAT_LON)

#(t): The model of the one-model figures, and the region of the regional ones
EXAMPLE_MODEL = 'CanESM5'
OBS_REGION = 'Antarctica (90–65°S)'

In [4]:
#(t): Where calc_03a to calc_03c saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

PosixPath('/work/scratch-pw4/aborow/ExtAnt')

## Which Results

Run **one** of the next two sections: each sets the folder the results are opened from, the sample's or the full data's.

### Sample data

In [5]:
#(t): The sample's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
RESULTS_DIR

PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/sample')

### Full data

In [6]:
#(t): The full data's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
RESULTS_DIR

PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/full')

## Open the Results

First what calc_03a to calc_03c saved in `RESULTS_DIR`, then each result opened.

In [7]:
#(t): What calc_03a to calc_03c saved in RESULTS_DIR
RESULTS_DIR, sorted(path.name for path in RESULTS_DIR.glob('*.zarr'))

(PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/full'),
 ['anomaly_ranks.zarr',
  'demo_anomaly_ranks.zarr',
  'demo_detrended_ranks.zarr',
  'demo_plume.zarr',
  'demo_statistics.zarr',
  'demo_tests.zarr',
  'detections.zarr',
  'detrended_ranks.zarr',
  'era5_kurtosis.zarr',
  'era5_lower_tail.zarr',
  'era5_mean.zarr',
  'era5_point.zarr',
  'era5_regional.zarr',
  'era5_regional_trend.zarr',
  'era5_skewness.zarr',
  'era5_std.zarr',
  'era5_trend.zarr',
  'era5_upper_tail.zarr',
  'era5_width.zarr',
  'eval_map_table.zarr',
  'eval_table.zarr',
  'experiment_noise.zarr',
  'final_mean.zarr',
  'final_quantiles.zarr',
  'first_look.zarr',
  'forced.zarr',
  'hist_nat_final_quantiles.zarr',
  'histnat_point.zarr',
  'historical_kurtosis.zarr',
  'historical_lower_tail.zarr',
  'historical_mean.zarr',
  'historical_point.zarr',
  'historical_skewness.zarr',
  'historical_std.zarr',
  'historical_trend.zarr',
  'historical_upper_tail.zarr',
  'historical_width.zarr',
  'mean_

In [31]:
#(t): Part 2 (calc_03c's): ERA5 among each experiment in OBS_REGION, /<model>/<experiment>
region_tests_tree = xr.open_datatree(RESULTS_DIR / 'region_tests.zarr', engine='zarr').load()

In [32]:
#(t): ... the record curves in OBS_REGION
record_curves_ds = xr.open_zarr(RESULTS_DIR / 'record_curves.zarr').load()

In [33]:
#(t): ... and at the point
point_curves_ds = xr.open_zarr(RESULTS_DIR / 'point_record_curves.zarr').load()

In [34]:
#(t): ... the detection maps
detections_ds = xr.open_zarr(RESULTS_DIR / 'detections.zarr').load()

In [35]:
#(t): ... the scaling factors of ANT and NAT
scaling_ant_nat_ds = xr.open_zarr(RESULTS_DIR / 'scaling_ant_nat.zarr').load()

In [36]:
#(t): ... and of the single forcings
scaling_single_ds = xr.open_zarr(RESULTS_DIR / 'scaling_single_forcings.zarr').load()

## 1. Consistency with Each Experiment

Each row is one experiment: its members' 1979–2013 trends (dots) and their 5–95% range (bar); the black line is ERA5's. Where the line crosses a row's bar (✓), that experiment could have produced the observed trend. ERA5 outside **hist-nat** is a detection of forced change; ERA5 inside **historical** is consistency.

In [ ]:
#(t): EXAMPLE_MODEL's experiments, each a dict entry of its ev.locate results on (statistic, season)
obs_model_results = {node.name: node.to_dataset() for node in region_tests_tree[EXAMPLE_MODEL].leaves if node.data_vars}

In [ ]:
#(t): The observed trend against each experiment's members' trends, one panel per season: ✓ where ERA5 is inside the
#(t): members' 5-95%
#(c): Gaps between the panels wide enough for each row's verdict, right of it
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2, panel_width=2.6,
                                                  left_margin=1.0, column_gap=1.5), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    ocp.plot_experiment_strips({experiment: results_ds.sel(season=season)
                                for experiment, results_ds in obs_model_results.items()}, 'trend', ax=ax)
    ax.set_title(season, loc='left')
    ax.set_xlabel(f"{ev.DIAGNOSTICS['trend'].label} ({ev.DIAGNOSTICS['trend'].units})")
strip_handles = ocp.experiment_strip_legend_handles()
legend_axes[0].legend(handles=strip_handles[:3], loc='center', ncols=3, frameon=False)
legend_axes[1].legend(handles=strip_handles[3:], loc='center', ncols=3, frameon=False)
fig.suptitle(f'Observed trend vs each experiment: {EXAMPLE_MODEL}, {OBS_REGION}', x=0.01, ha='left');

In [ ]:
#(t): The same for the S/N against itself (smoothed change over the record / its own noise)
#(c): Gaps between the panels wide enough for each row's verdict, right of it
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2, panel_width=2.6,
                                                  left_margin=1.0, column_gap=1.5), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    ocp.plot_experiment_strips({experiment: results_ds.sel(season=season)
                                for experiment, results_ds in obs_model_results.items()}, 'emergence', ax=ax)
    ax.set_title(season, loc='left')
    ax.set_xlabel(ev.DIAGNOSTICS['emergence'].label)
legend_axes[0].legend(handles=strip_handles[:3], loc='center', ncols=3, frameon=False)
legend_axes[1].legend(handles=strip_handles[3:], loc='center', ncols=3, frameon=False)
fig.suptitle(f'Observed S/N vs each experiment: {EXAMPLE_MODEL}, {OBS_REGION}', x=0.01, ha='left');

In [ ]:
#(t): Every model's trend verdicts only (no members), the experiments as the scorecard's columns
trend_verdicts_tree = xr.DataTree.from_dict({
    node.path: node.to_dataset()[['percentile', 'verdict', 'testable']].sel(statistic='trend', drop=True)
    for node in region_tests_tree.leaves
})
trend_table_ds = tree_to_dataset(trend_verdicts_tree, dims=['model', 'experiment'])
trend_table_ds = trend_table_ds.rename(experiment='statistic')
trend_table_ds

In [ ]:
#(t): ERA5's 1979-2013 trend among each experiment's members: every model (rows), experiment and season (columns)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, n_colorbars=1,
                                                  panel_width=0.44 * trend_table_ds.sizes['statistic'] * len(SEASONS),
                                                  panel_height=0.46 * trend_table_ds.sizes['model'], left_margin=1.5),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
trend_scorecard_image = evp.plot_scorecard(trend_table_ds, ax=axes[0, 0])
fig.colorbar(trend_scorecard_image, ax=axes, location='bottom', shrink=0.6, ticks=evp.PERCENTILE_BOUNDS,
             label='ERA5 percentile among the members')
fig.suptitle(f"ERA5's 1979-2013 trend among each experiment's members ({OBS_REGION})", x=0.01, ha='left');

## 2. Record Rates

Cumulative record highs and lows through 1979–2013, with the four seasons pooled so there are enough records to compare. The dashed curve is what a climate with no change gives (1 + 1/2 + … + 1/*n* per season). Record highs arriving faster than that, and faster than the hist-nat members', mean warming the natural range cannot produce; record lows arriving slower mean the cold end has moved too.

In [ ]:
#(t): Cumulative record highs and lows in OBS_REGION, all seasons pooled: ERA5 against chance (dashed) and against
#(t): hist-nat's and historical's members (median and 5-95%)
record_kinds = list(record_curves_ds['kind'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(record_kinds), n_legends=1, panel_width=4.5,
                                                  panel_height=3.0), layout='constrained')
axes = fig.add_gridspec(1, len(record_kinds)).subplots(squeeze=False)
for ax, kind in zip(axes[0], record_kinds):
    ocp.plot_record_rates(record_curves_ds, kind, ax=ax)
    ax.set_title(f'Record {kind}', loc='left')
    ax.set_xlabel('Year')
axes[0, 0].set_ylabel('Cumulative number of records')
fig.legend(handles=ocp.record_rate_legend_handles(), loc='outside lower center', ncols=4, frameon=False)
fig.suptitle(f'Record rates, all seasons pooled: {EXAMPLE_MODEL}, {OBS_REGION}', x=0.01, ha='left');

In [ ]:
#(t): At the point (much noisier than a regional mean)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(record_kinds), n_legends=1, panel_width=4.5,
                                                  panel_height=3.0), layout='constrained')
axes = fig.add_gridspec(1, len(record_kinds)).subplots(squeeze=False)
for ax, kind in zip(axes[0], record_kinds):
    ocp.plot_record_rates(point_curves_ds, kind, ax=ax)
    ax.set_title(f'Record {kind}', loc='left')
    ax.set_xlabel('Year')
axes[0, 0].set_ylabel('Cumulative number of records')
fig.legend(handles=ocp.record_rate_legend_handles(), loc='outside lower center', ncols=4, frameon=False)
fig.suptitle(f'Record rates, all seasons pooled: {EXAMPLE_MODEL}, {POINT_LABEL}', x=0.01, ha='left');

## 3. Detection Maps

The trend test at every grid point: detected (outside hist-nat's 5–95%) and consistent (inside historical's). At p < 0.1 about 10% of points land outside a range by chance; read coherent regions.

In [ ]:
#(t): Is the observed (ERA5) trend detectable, and is historical consistent with it: every model (rows) and season, the
#(t): detection classes (oc.DETECTION_CLASSES); a map with no class at all is an ensemble too small to test
detection_class_da = detections_ds['detection_class'].sel(season=SEASONS)
models = detection_class_da['model'].values
class_names = [name.replace(', ', ',\n') for name in oc.DETECTION_CLASSES.values()]
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(detection_class_da, 'model', 'season', axes=axes, cax=cax,
                                  levels=np.arange(-0.5, len(ocp.DETECTION_COLORS)), cmap=ListedColormap(ocp.DETECTION_COLORS),
                                  extend='neither', ticks=range(len(ocp.DETECTION_COLORS)), ticklabels=class_names,
                                  label="Detected: outside hist-nat's 5–95%. Consistent: inside historical's 5–95%.")
#(c): Say so where a map has no class at all, rather than leave it blank
for i, model in enumerate(models):
    for j, season in enumerate(SEASONS):
        if bool(detection_class_da.sel(model=model, season=season).isnull().all()):
            axes[i, j].text(0.5, 0.5, 'too few members\nto test (N < 20)', transform=axes[i, j].transAxes,
                            ha='center', va='center', fontsize=9)
antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, models)
fig.suptitle('Is the observed (ERA5) trend detectable, and is historical consistent with it?');

In [ ]:
#(t): How many models call each point detected and consistent (class 1), every season (white for none)
n_detection_models = detections_ds.sizes['model']
detection_counts_da = (detections_ds['detection_class'] == 1).sum('model').sel(season=SEASONS)
detection_count_cmap, _ = rcp.count_colormap(n_detection_models, 'Blues')
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=1, n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=1, n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(detection_counts_da, None, 'season', axes=axes, cax=cax,
                                  levels=np.arange(-0.5, n_detection_models + 1.5), cmap=detection_count_cmap,
                                  extend='neither', ticks=range(n_detection_models + 1), label='Number of models')
antarctica.label_columns(axes, SEASONS)
fig.suptitle(f'Number of models where the observed trend is {oc.DETECTION_CLASSES[1]}');

## 4. Scaling Factors: How Much of Each Response Is Observed

β = 1 (the dashed line): the model's response has the observed size; an interval clear of 0 is a detection. The grey tick is the perfect-model median, which shows any bias in the method; it should be close to 1.

In [ ]:
#(t): Two signals, anthropogenic (ANT) and natural (NAT): their scaling factors in each region (panels), every model
#(t): (marker) and signal (colour); β = 1 dashed, β = 0 grey
scaling_regions = list(scaling_ant_nat_ds['region'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(scaling_regions), n_legends=1),
                 layout='constrained')
axes = fig.add_gridspec(1, len(scaling_regions)).subplots(sharey=True, squeeze=False)
for ax, region in zip(axes[0], scaling_regions):
    ocp.plot_scaling_factors(scaling_ant_nat_ds.sel(region=region), ax=ax)
    ax.set_title(region, loc='left')
axes[0, 0].set_ylabel('Scaling factor β')
scaling_handles = ocp.scaling_factor_legend_handles(scaling_ant_nat_ds['model'].values)
fig.legend(handles=scaling_handles, loc='outside lower center', ncols=len(scaling_handles), frameon=False)
fig.suptitle('Scaling factors: anthropogenic (ANT) and natural (NAT)', x=0.01, ha='left');

In [ ]:
#(t): ... and how much of the observed trend each explains (β × the signal's trend), every model (rows)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(scaling_regions), n_legends=1,
                                                  panel_width=3.6, left_margin=1.5), layout='constrained')
axes = fig.add_gridspec(1, len(scaling_regions)).subplots(sharey=True, squeeze=False)
for ax, region in zip(axes[0], scaling_regions):
    ocp.plot_attributable_trends(scaling_ant_nat_ds.sel(region=region), ax=ax)
    ax.set_title(region, loc='left')
    ax.set_xlabel('Trend 1979–2014 (°C/decade)')
attributable_handles = ocp.attributable_trend_legend_handles(scaling_ant_nat_ds['signal'].values)
fig.legend(handles=attributable_handles, loc='outside lower center', ncols=len(attributable_handles), frameon=False)
fig.suptitle('How much of the observed trend ANT and NAT explain', x=0.01, ha='left');

In [ ]:
#(t): Four signals, greenhouse gases, aerosols, ozone and natural: their scaling factors
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(scaling_regions), n_legends=1),
                 layout='constrained')
axes = fig.add_gridspec(1, len(scaling_regions)).subplots(sharey=True, squeeze=False)
for ax, region in zip(axes[0], scaling_regions):
    ocp.plot_scaling_factors(scaling_single_ds.sel(region=region), ax=ax)
    ax.set_title(region, loc='left')
axes[0, 0].set_ylabel('Scaling factor β')
scaling_handles = ocp.scaling_factor_legend_handles(scaling_single_ds['model'].values)
fig.legend(handles=scaling_handles, loc='outside lower center', ncols=len(scaling_handles), frameon=False)
fig.suptitle('Scaling factors: single forcings', x=0.01, ha='left');

In [ ]:
#(t): ... and how much of the observed trend each explains
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(scaling_regions), n_legends=1,
                                                  panel_width=3.6, left_margin=1.5), layout='constrained')
axes = fig.add_gridspec(1, len(scaling_regions)).subplots(sharey=True, squeeze=False)
for ax, region in zip(axes[0], scaling_regions):
    ocp.plot_attributable_trends(scaling_single_ds.sel(region=region), ax=ax)
    ax.set_title(region, loc='left')
    ax.set_xlabel('Trend 1979–2014 (°C/decade)')
attributable_handles = ocp.attributable_trend_legend_handles(scaling_single_ds['signal'].values)
fig.legend(handles=attributable_handles, loc='outside lower center', ncols=len(attributable_handles), frameon=False)
fig.suptitle('How much of the observed trend each forcing explains', x=0.01, ha='left');